# Check map and layout names

Read-only audit of an ArcGIS Pro project. It reports names that are known to
cause problems and **changes nothing** - no renames, no deletes, no save.

| Check | Why it matters |
|---|---|
| Map with an empty name | Crashes ArcGIS Pro when a layout view builds its map drop-down |
| Duplicate map names | Sign the layout tools have been re-run and left copies behind |
| Maps no layout uses | Cleanup candidates - delete with care, bookmarks and reports can reference them |
| Layout with an empty name | Makes the layout tools produce an unnamed map |
| Illegal characters in a layout name | Layout names become the `.mapx` / `.png` / `.pagx` file names, so the export fails with only a warning |
| Map frame with no map | A frame that will not draw |

This also works on a project whose layouts crash ArcGIS Pro, because it reads
the project through arcpy instead of through the user interface.

In [1]:
import arcpy
from collections import Counter

### Choose the project

Leave `APRX_PATH` as `"CURRENT"` to check the project you have open. To check a
different project - including one you cannot open properly - put the full path
to its `.aprx` instead.

In [2]:
APRX_PATH = "CURRENT"

# Example of checking a project on disk instead:
# APRX_PATH = r"C:\workspace\Some project\Some project.aprx"

### Run as is

In [3]:
# Characters Windows forbids in a file name. The layout tools build their
# .mapx / .png / .pagx file names from the layout name, so any of these makes
# that export fail - and it is only reported as a warning.
ILLEGAL_FILENAME_CHARS = '<>:"/\\|?*'

RESERVED_DEVICE_NAMES = {'CON', 'PRN', 'AUX', 'NUL'}
RESERVED_DEVICE_NAMES |= {f'COM{i}' for i in range(1, 10)}
RESERVED_DEVICE_NAMES |= {f'LPT{i}' for i in range(1, 10)}

LONG_NAME_THRESHOLD = 100


def is_blank(name):
    return not (name or '').strip()


def filename_problems(name):
    """Reasons this name would be unsafe to use as a file name."""
    found = []
    bad = sorted({c for c in name if c in ILLEGAL_FILENAME_CHARS})
    if bad:
        found.append('illegal file name character(s): ' + ' '.join(bad))
    if name != name.rstrip(' .'):
        found.append('ends with a space or a dot')
    if name.split('.')[0].strip().upper() in RESERVED_DEVICE_NAMES:
        found.append('reserved Windows device name')
    if len(name) > LONG_NAME_THRESHOLD:
        found.append(f'{len(name)} characters long - risks the 260 character path limit')
    return found


aprx = arcpy.mp.ArcGISProject(APRX_PATH)
maps = aprx.listMaps()
layouts = aprx.listLayouts()

critical, warnings, notes = [], [], []

# --- Map frames: which maps are used, and which frames point at nothing ------
used_map_names = set()
for lyt in layouts:
    try:
        frames = lyt.listElements('MAPFRAME_ELEMENT')
    except Exception as e:
        warnings.append(f'Could not read the map frames of layout "{lyt.name}": {e}')
        continue
    for mf in frames:
        try:
            frame_map = mf.map
        except Exception as e:
            warnings.append(f'Map frame "{mf.name}" in layout "{lyt.name}" could not be read: {e}')
            continue
        if frame_map is None:
            warnings.append(f'Map frame "{mf.name}" in layout "{lyt.name}" has no map assigned.')
        else:
            used_map_names.add(frame_map.name)

# --- Maps --------------------------------------------------------------------
map_names = [mp.name for mp in maps]

for i, name in enumerate(map_names):
    if is_blank(name):
        critical.append(
            f'The map at position {i} has an empty name. This is what crashes '
            'ArcGIS Pro when a layout view builds its map drop-down.')

for name, count in Counter(map_names).items():
    if count > 1 and not is_blank(name):
        warnings.append(f'{count} maps share the name "{name}".')

for name in sorted(set(map_names)):
    if not is_blank(name) and name not in used_map_names:
        notes.append(f'No layout uses the map "{name}".')

if layouts and len(maps) > 3 * len(layouts):
    notes.append(
        f'{len(maps)} maps for {len(layouts)} layouts. A ratio this high usually '
        'means the layout tools have been re-run and left copies behind.')

# --- Layouts -----------------------------------------------------------------
layout_names = [lyt.name for lyt in layouts]

for i, name in enumerate(layout_names):
    if is_blank(name):
        critical.append(f'The layout at position {i} has an empty name.')
        continue
    for problem in filename_problems(name):
        warnings.append(f'Layout "{name}": {problem}')

for name, count in Counter(layout_names).items():
    if count > 1 and not is_blank(name):
        warnings.append(f'{count} layouts share the name "{name}".')

# --- Report ------------------------------------------------------------------
print(f'Project : {aprx.filePath}')
print(f'Maps    : {len(maps)}')
print(f'Layouts : {len(layouts)}')

for heading, items in (('CRITICAL', critical), ('WARNING', warnings), ('NOTE', notes)):
    print(f'\n{heading}  ({len(items)})')
    if not items:
        print('    nothing found')
    for item in items:
        print(f'    - {item}')

if not critical and not warnings:
    print('\nNo problems found.')

# Release the lock so ArcGIS Pro can still open a project read from disk
if APRX_PATH != 'CURRENT':
    del aprx

Project : C:\workspace\Freelance life\Pantelis Karapatsios\ArcGIS Workflow Optimization\Orthophoto ArcPy automation\Orthophoto ArcPy automation.aprx
Maps    : 31
Layouts : 3

CRITICAL  (0)
    nothing found

WARNING  (1)
    - 3 maps share the name "Εικόνα 1_ Μορφές Δασικού Χάρτη και Γεωτεμαχίου ευρύτερης περιοχής".

NOTE  (25)
    - No layout uses the map "20_05_test_1".
    - No layout uses the map "20_05_test_2".
    - No layout uses the map "20_05_test_3".
    - No layout uses the map "20_05_test_4".
    - No layout uses the map "5aria".
    - No layout uses the map "Layout 4".
    - No layout uses the map "check label_no".
    - No layout uses the map "check_label_yes".
    - No layout uses the map "test 1".
    - No layout uses the map "test 2".
    - No layout uses the map "test 3".
    - No layout uses the map "test layout for legend".
    - No layout uses the map "test transp".
    - No layout uses the map "test_transp".
    - No layout uses the map "ΓΕΩΤΕΜΑΧΙΟ & ΕΠΙΔΙΚΟ ΤΜΗΜΑ

### What the results mean

- **CRITICAL** - fix before working in the layouts. An empty map name is the
  condition that crashes ArcGIS Pro when a layout view builds its map list.
- **WARNING** - causes silent failures or confusion, but will not crash Pro. An
  illegal character in a layout name makes the layout tools skip that layout's
  `.mapx` / `.png` / `.pagx` export with only a warning in the tool messages.
- **NOTE** - informational only. Unused maps are safe to leave where they are;
  deleting them is a separate decision, because bookmarks and reports can still
  reference a map that no layout draws.

One limitation: if several maps share a name, the "no layout uses" check is
approximate. It matches on the name, so it cannot tell two identically named
maps apart.